# ClimaTwin India — Train the 2020 real-INSAT-3D-LST model (Colab GPU)

Trains the **focused one-year (2020) ConvLSTM** on the cube that carries **real INSAT-3D
Land Surface Temperature** (one ~0600 UTC overpass/day, from MOSDAC), fused with IMD
rainfall/Tmax/Tmin. Strictly temporal month split: **train Jan-Sep / val Oct / test Nov-Dec**;
normalization on train months only (no leakage).

This is the **parallel `insat_real` regime** — the validated synthetic 2000-2023 model is
untouched. Output: `models/checkpoints/convlstm_2020.pt`, which you download and drop into
your local `models/checkpoints/` to flip the dashboard's INSAT-3D regime from PENDING to ACTIVE.

**Honesty note:** one year is little data and persistence is a strong 1-day baseline; the
final cell reports the real skill-vs-persistence on the held-out Nov-Dec test months — read it
as-is, do not over-claim.


## 1. Check the GPU


In [ ]:
!nvidia-smi


In [ ]:
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')


## 2. Get the code + the 2020 cube
Build the bundle locally first with the prebuilt real-LST cube included:
```bash
bash scripts/make_colab_bundle.sh --with-data   # includes data/twin_cube_2020.nc
```
Then upload `climatwin_bundle.zip` below. (Or use option-a to clone the repo, but the
cube is gitignored, so you must still upload the cube separately.)


In [ ]:
# Option (a) — clone the repo (code only; the cube is gitignored and must still be uploaded):
# !git clone https://github.com/ayushap18/climatwin-india.git climatwin
# %cd climatwin


In [ ]:
# Option (b) DEFAULT — upload climatwin_bundle.zip (built with --with-data):
from google.colab import files
uploaded = files.upload()   # choose climatwin_bundle.zip
!unzip -q -o climatwin_bundle.zip -d climatwin
%cd climatwin
!ls data/*2020* 2>/dev/null || echo 'NOTE: data/twin_cube_2020.nc not found — rebuild the bundle with --with-data'


## 3. Install dependencies


In [ ]:
!pip install -q xarray netCDF4 cftime


## 4. Verify the real-LST 2020 cube


In [ ]:
import xarray as xr, json
d = xr.open_dataset('data/twin_cube_2020.nc')
print('regime      :', d.attrs.get('regime'))
print('lst_source  :', d.attrs.get('lst_source'), '| real coverage:', d.attrs.get('lst_coverage'),
      '| real days:', d.attrs.get('lst_real_days'))
print('vars        :', list(d.data_vars), '| days:', d.sizes['time'])
print('split       :', json.loads(d.attrs.get('split_dates')))
d.close()


## 5. Train the 2020 ConvLSTM (real LST channel)
Two-head rainfall (P(rain)+amount) + temperature MSE, AdamW + cosine LR, early stopping.
GPU runs ~80 epochs in well under a minute on this tiny 9x13 grid.


In [ ]:
!python -m models.train \
    --cube data/twin_cube_2020.nc \
    --norm data/norm_stats_2020.json \
    --out convlstm_2020.pt \
    --epochs 120 --hidden 64 --layers 2 --seed 0


## 6. Honest skill check — ConvLSTM vs persistence on the held-out test months
Persistence = 'tomorrow equals today'. We score RMSE on the **Nov-Dec** test window for
Tmax/Tmin (degC) and rainfall (mm). One year is little data, so report whatever this shows.


In [ ]:
import json, numpy as np, pandas as pd, xarray as xr, torch
from models.convlstm import build_input, build_module, elevation_stats, in_channels, norm_var
import config as cfg

d = xr.open_dataset('data/twin_cube_2020.nc'); norm = json.load(open('data/norm_stats_2020.json'))
split = norm['_split_dates']; t0, t1 = split['test']
VARS = cfg.VARS; k = cfg.K_INPUT
elev_da = d['elevation']; elev = (elev_da.isel(time=0) if 'time' in elev_da.dims else elev_da).values.astype('float32')
es = elevation_stats(elev)
ck = torch.load('models/checkpoints/convlstm_2020.pt', map_location='cpu', weights_only=False)
m = build_module(**ck['arch']); m.load_state_dict(ck['state_dict']); m.eval()

times = pd.to_datetime(d['time'].values)
raw = np.stack([d[v].values for v in VARS], axis=1).astype('float32')   # (T,3,H,W)
lst = d['lst'].values.astype('float32')
def denorm(x, st):
    x = x*st['std']+st['mean']
    return np.expm1(x) if st['transform']=='log1p' else x
test_idx = [t for t in range(k, len(raw)) if t0 <= str(times[t])[:10] <= t1]
errs = {v: {'cl':[], 'pe':[]} for v in VARS}
for t in test_idx:
    xb = build_input(raw[t-k:t], times[t-k:t], elev, norm, es, lst[t-k:t])[None]
    with torch.no_grad(): out = m(torch.from_numpy(xb))[0].numpy()
    # two-head: rainfall = P(rain)*amount; channels [p, amount, tmax, tmin]
    pred = {}
    if ck['arch']['out_ch']==4:
        p = 1/(1+np.exp(-out[0])); amt = denorm(out[1], norm['rainfall'])
        pred['rainfall'] = np.clip(p,0,1)*np.clip(amt,0,None)
        pred['tmax']=denorm(out[2],norm['tmax']); pred['tmin']=denorm(out[3],norm['tmin'])
    else:
        pred['rainfall']=denorm(out[0],norm['rainfall']); pred['tmax']=denorm(out[1],norm['tmax']); pred['tmin']=denorm(out[2],norm['tmin'])
    for vi,v in enumerate(VARS):
        truth = raw[t, vi]; persist = raw[t-1, vi]
        errs[v]['cl'].append(np.nanmean((pred[v]-truth)**2)); errs[v]['pe'].append(np.nanmean((persist-truth)**2))
print(f'2020 INSAT-3D regime — test {t0}..{t1}  ({len(test_idx)} days)')
print(f"{'var':9} {'ConvLSTM RMSE':>14} {'Persistence RMSE':>17} {'winner':>10}")
for v in VARS:
    cl=float(np.sqrt(np.mean(errs[v]['cl']))); pe=float(np.sqrt(np.mean(errs[v]['pe'])))
    print(f'{v:9} {cl:14.3f} {pe:17.3f} {("ConvLSTM" if cl<pe else "persistence"):>10}')
print('\nHonest read: on one year, beating persistence at 1-day is hard, especially for the dry-winter rainfall test.')


## 7. Download the checkpoint
Drop `convlstm_2020.pt` into your local `models/checkpoints/`. The backend auto-detects it and
the dashboard's INSAT-3D regime flips from **PENDING** to **ACTIVE**.


In [ ]:
from google.colab import files
files.download('models/checkpoints/convlstm_2020.pt')
